<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/LLM%20Agents%20Alapok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Agents Alapok

**Licenc: CC BY-NC-SA 4.0**

Az **LLM Agent** egy autonóm rendszer, ahol az LLM **döntéseket hoz**, **eszközöket használ**, és **iteratívan** dolgozik a feladat megoldásáig.

## Agent = LLM + Tools + Memory

```
User Query → [LLM] → Think → Act → Observe → ... → Final Answer
                ↑                    ↓
             Memory              Tools (Search, Calculator, API, ...)
```

## Miért kellenek agent-ek?

Az LLM-ek önmagukban **korlátozottak**:
- ❌ Nem tudnak interneten keresni
- ❌ Nincs aktuális tudásuk
- ❌ Nem tudnak számolni megbízhatóan
- ❌ Nem tudnak fájlokat kezelni

Az **agent** megoldja ezeket: az LLM **orchestrál**, és a tool-ok **végrehajtanak**.

## Tartalomjegyzék
1. ReAct Pattern
2. Tool-use Agent
3. Function Calling (OpenAI style)
4. Agent Frameworks
5. Agent típusok és architektúrák

In [ ]:
import json
from typing import Dict, List, Callable

## 1. ReAct Pattern

A **ReAct** (Yao et al., 2022) = **Re**asoning + **Act**ing. Az agent váltakozva gondolkodik és cselekszik.

### ReAct loop

```
1. Thought: Mit kellene tennem? (reasoning)
2. Action: Tool hívás (acting)
3. Observation: Tool eredménye (feedback)
4. ... ismétlés amíg kész ...
5. Final Answer: Végső válasz
```

### Miért működik?

- **Interleaved reasoning**: Minden lépés előtt gondolkodik
- **Grounded**: A tool eredmények "földelik" a gondolkodást
- **Traceable**: Látható a döntési folyamat

In [ ]:
# Szimulált ReAct loop
react_example = """
Question: Mi a népessége Magyarország fővárosának?

Thought: Először meg kell tudnom, mi Magyarország fővárosa.
Action: search("Magyarország fővárosa")
Observation: Budapest Magyarország fővárosa.

Thought: Most meg kell keresnem Budapest népességét.
Action: search("Budapest népessége")
Observation: Budapest népessége kb. 1.7 millió fő.

Thought: Megvan a válasz.
Final Answer: Budapest népessége körülbelül 1.7 millió fő.
"""
print(react_example)

## 2. Tool-use Agent

Az agent **tool-okat** hív, amelyek külső műveleteket hajtanak végre.

### Tipikus tool-ok

| Tool | Funkció | Példa |
|------|---------|-------|
| **Search** | Web keresés | Google, Bing |
| **Calculator** | Matematikai műveletek | Pontos számítások |
| **Code Interpreter** | Kód futtatás | Python REPL |
| **API** | Külső szolgáltatások | Weather, Stock |
| **Database** | Adatlekérdezés | SQL |
| **File I/O** | Fájlkezelés | Olvasás, írás |

### Tool definition

Minden tool-hoz tartozik:
- **Név**: Egyedi azonosító
- **Leírás**: Mire való (LLM számára)
- **Paraméterek**: JSON schema

In [ ]:
class SimpleAgent:
    def __init__(self):
        self.tools = {
            "calculator": self.calculator,
            "search": self.search,
            "weather": self.weather,
        }
        self.memory = []

    def calculator(self, expr: str) -> str:
        try:
            return str(eval(expr))
        except:
            return "Error"

    def search(self, query: str) -> str:
        db = {"python": "Python egy programozási nyelv.",
              "budapest": "Budapest Magyarország fővárosa."}
        return db.get(query.lower(), "Nincs találat.")

    def weather(self, city: str) -> str:
        return f"{city}: 15°C, napos"

    def run(self, query: str) -> str:
        self.memory.append({"role": "user", "content": query})

        # Egyszerű tool routing (valódi agent LLM-et használna)
        if "számol" in query or "+" in query or "*" in query:
            expr = query.split(":")[-1].strip() if ":" in query else "2+2"
            result = self.tools["calculator"](expr)
            return f"Eredmény: {result}"
        elif "keres" in query:
            term = query.split(":")[-1].strip()
            return self.tools["search"](term)
        elif "idő" in query:
            city = "Budapest"
            return self.tools["weather"](city)
        return "Nem tudom megválaszolni."

agent = SimpleAgent()
print(agent.run("Számold ki: 15 * 7"))
print(agent.run("Keress: python"))
print(agent.run("Milyen idő van?"))

## 3. Function Calling (OpenAI style)

Az OpenAI API natívan támogatja a **function calling**-ot, ahol az LLM strukturált JSON-t generál a tool híváshoz.

### Flow

```
1. User: "Milyen idő van Budapesten?"
2. LLM: {"tool": "get_weather", "args": {"city": "Budapest"}}
3. System: Tool hívás → "15°C, napos"
4. LLM: "Budapesten 15°C van és napos az idő."
```

### Előnyök

- **Strukturált output**: JSON, nem szabad szöveg
- **Reliable**: Kevesebb parsing hiba
- **Parallel**: Több tool egy válaszban

In [ ]:
# Tool definition
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"}
                },
                "required": ["city"]
            }
        }
    }
]

print("Tool definition:")
print(json.dumps(tools, indent=2))

In [ ]:
# Teljes function calling flow
print("""
from openai import OpenAI

client = OpenAI()

# 1. LLM eldönti, melyik tool-t hívja
response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": "Milyen idő van Budapesten?"}],
    tools=tools
)

# 2. Tool hívás
tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = get_weather(args["city"])  # Saját implementáció

# 3. Eredmény visszaadása LLM-nek
messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
final = client.chat.completions.create(model="gpt-4", messages=messages)
""")

## 4. Agent Frameworks

Több framework segíti az agent fejlesztést:

### Népszerű frameworkök

| Framework | Előny | Use case |
|-----------|-------|----------|
| **LangChain** | Flexibilis, sok integráció | Általános |
| **LlamaIndex** | RAG fókusz | Dokumentum Q&A |
| **AutoGPT** | Autonóm | Kísérleti |
| **CrewAI** | Multi-agent | Csapat szimuláció |
| **Semantic Kernel** | Microsoft, enterprise | .NET/Python |

In [ ]:
print("""
# LangChain Agent
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import Tool

tools = [
    Tool(name="Search", func=search_fn, description="Web search"),
    Tool(name="Calculator", func=calc_fn, description="Math")
]

agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools)
result = executor.invoke({"input": "What is 25 * 4?"})
""")

## 5. Agent típusok és architektúrák

### Agent típusok

| Típus | Leírás | Mikor használd |
|-------|--------|----------------|
| **ReAct** | Reason + Act iteráció | Általános, lépésről lépésre |
| **Plan-and-Execute** | Előre tervez, majd végrehajt | Komplex, több lépéses feladatok |
| **Multi-agent** | Több specializált agent | Komplex rendszerek |
| **RAG Agent** | Retrieval + Generation | Dokumentum alapú Q&A |
| **Reflexion** | Self-reflection és javítás | Magas minőség kell |

### Agent vs Chain

| Chain | Agent |
|-------|-------|
| Fix lépések | Dinamikus döntések |
| Determinisztikus | LLM dönt |
| Egyszerű | Komplex |

### Biztonsági megfontolások

⚠️ Az agent-ek **autonóm döntéseket** hoznak!

- **Sandbox**: Korlátozott környezet
- **Rate limiting**: Tool hívások limitálása
- **Human-in-the-loop**: Kritikus műveleteknél jóváhagyás
- **Logging**: Minden action naplózása

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

# Agent loop
boxes = [
    (0.1, 0.5, 'User\nQuery', 'lightblue'),
    (0.3, 0.5, 'LLM\n(Think)', 'lightgreen'),
    (0.5, 0.5, 'Tool\n(Act)', 'lightyellow'),
    (0.7, 0.5, 'Result\n(Observe)', 'lightcoral'),
    (0.9, 0.5, 'Final\nAnswer', 'plum'),
]

for x, y, text, color in boxes:
    ax.add_patch(plt.Rectangle((x-0.08, y-0.15), 0.16, 0.3, facecolor=color, edgecolor='black'))
    ax.text(x, y, text, ha='center', va='center', fontsize=9)

# Arrows
for i in range(4):
    ax.annotate('', xy=(0.24 + i*0.2, 0.5), xytext=(0.18 + i*0.2, 0.5),
               arrowprops=dict(arrowstyle='->'))

# Loop back
ax.annotate('', xy=(0.3, 0.3), xytext=(0.7, 0.3),
           arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=-0.3', color='green'))
ax.text(0.5, 0.2, 'iterate until done', ha='center', fontsize=8, color='green')

plt.title('LLM Agent Loop', fontsize=12, fontweight='bold')
plt.show()

## Összefoglalás

### Főbb tanulságok

1. **Agent = LLM + Tools + Memory** - kiterjeszti az LLM képességeit
2. **ReAct pattern**: Gondolkodás és cselekvés iterációja
3. **Function calling**: Strukturált tool invokáció
4. **Frameworks**: LangChain, LlamaIndex segítenek

### Agent design patterns

| Pattern | Leírás |
|---------|--------|
| **Tool selection** | Melyik tool-t hívjuk? |
| **Multi-step planning** | Komplex feladatok lebontása |
| **Error recovery** | Mit tegyünk, ha hibázik? |
| **Memory management** | Hosszú beszélgetések kezelése |

### Best practices

1. **Kezdj egyszerűen**: Kevés, jól definiált tool
2. **Jó tool descriptions**: Az LLM ebből "érti" a tool-t
3. **Fallback**: Ha az agent megakad
4. **Tesztelj sokat**: Edge case-ek
5. **Monitor**: Production-ben naplózd az action-öket

### Következő lépések

A következő notebookban az **RLHF** (Reinforcement Learning from Human Feedback) témát nézzük, ami az LLM-ek "igazítását" teszi lehetővé emberi preferenciákhoz!